# A GMSH Manual

We describe first the use of GMSH for the generation of the geometry and the generation of the geometry. We subsequently discuss how to interate over elements in the mesh that GMSH generates. 

For the **geometry generation**, GMSH can be employed using one of three methods that are described next: 
1. The first method is to build the geometry in GMSH from the ground up using the GMSH primitives for points, lines, surfaces and volumes. This allows to generate 1D, 2D or 3D geometries. This way of working provides detailed control over the geometry definition. It allows to automate the geometry creation through scripting. Two examples are given later in this notebook.  A possible disadvantage is the large overhead for complex geometries. 
2. The second method is create the geometry in dedicated CAD modeling tools, to save the geometry to file and to import the geometry into the GMSH graphical user interface. Examples of public domain CAD modeling tools include [Blender](https://www.blender.org), [FreeCAD](https://www.freecad.org) and [Salome](https://www.salome-platform.org). 
3. The third method is to use GMSH as a plugin for CAD tools as e.g. a [plugin for Blender](https://github.com/blender-for-science/blendmsh), [plugin for FreeCAD](https://wiki.freecad.org/Macro_GMSH) or [plugin for SALOME](https://docs.salome-platform.org/latest/gui/GMSHPLUGIN/index.html). 

For the (un)structured **mesh generation** in 1D, 2D and 3D, GMSH provides various algorithms that are described in the GMSH manual at [GMSH Mesh Module](https://gmsh.info/doc/texinfo/gmsh.html#Choosing-the-right-unstructured-algorithm). In 2D, a mesh consisting of triangles is called a [triangulation](https://en.wikipedia.org/wiki/Triangulation_(geometry)). Often a variant of the [Delaunay algorithm](https://en.wikipedia.org/wiki/Delaunay_triangulation) is used to construct these triangular meshes. We refer to [Mesh Generation](https://en.wikipedia.org/wiki/Mesh_generation) for more information on mesh generation.

## Import Packages 

In [1]:
try
    using Gmsh: Gmsh, gmsh
catch
    using gmsh
end 

using GR 
using LinearAlgebra
using SparseArrays

using Test 

using Plots

## Section 1: Introduction 

More later.  

## Section2: Low-Level Geometry Definition using GMSH Primitives 

Here we generate the mesh using low-level GMSH primitives. 

### Section 1.2: An Small Example 

Geometry definition and mesh generation of square domain where $0 \leq x \leq 1$ and $0 \leq y \leq 1$. 

The code that follows performs the followings steps. First the <b>geometry</b> on the unit square geometry is defined, then the geometry model is synchronized, then physical groups are added and finally the <b>mesh</b> is generated. 

<b>Geometry Definition</b> 

First the <b>geometry</b> is generated in the following five steps:
1. four corner points of the square are defined. The points are defined by their $(x,y,z)$-coordinates. For two-dimensional geometries, the $z$-coordinate set to zero. The points are labeled as 1 through 4. See docs of function <i>gmsh.model.geo.addPoint()</i> for more information;
2. four lines are defined as the edges of the square are defined by connecting previously defined points. Edges are formed by connecting points pairwise. The lines are given a start and end point. The lines are thus oriented. GMSH distinguishes between tags given to points (entities of dimensions zero), tags given to lines (entities of dimension one), tags given to surfaces (entities of dimension two) and tags given to volumes (entities of dimension three). GMSH uses the convention (dim,tag) to distinguish various entities. The edges are labeled as 1 through 4. See docs of function <i>gmsh.model.geo.addLine()</i> for more information;  
3. the boundary of the square is defined by a loop connecting the four edges. The orientation of the edges given an orientation to the loop. The loop is oriented such that an imaginary observer walking on the loop finds the domain on his left-hand side. The loop is labeled as 1. See docs of function <i>gmsh.model.geo.addCurveLoop()</i> for more information;   
4. the surface of the square is defined by the loop. It is on this square that the mesh generation will take place. This square is labeled as 1. See docs of function <i>gmsh.model.geo.addPlaneSurface()</i> for more information;

<b>Remark on labeling entities</b> 

GMSH distinguishes points, lines, surfaces and volumes as entities of dimension 0, 1, 2 and 3, respectively. GMSH recognizes entities using (dimensions,label)-pairs. Entities of the same dimension therefore should have unique labels. Nothing, however, prevents entities with different dimension to share the same label;

After the geometry definition, the geometry model is synchronized using the function <i>gmsh.model.geo.synchronize()</i> and physical groups for the boundary and the interior of the domain are added to the model using the function <i>gmsh.model.addPhysicalGroup</i>. 

<b>Mesh Generation</b>

Next the <b>mesh</b> on the geometry is defined by mesh generation using the function <i>gmsh.model.mesh.generate(2)</i>, where the input refers to two-dimensional mesh generation. By default, the mesh is generated by first meshing the four edges of the square. The mesh is subsequently propagated towards the interior of the square. The mesh density is controlled by the parameter lc. 

The mesh is optionally written to an output file and visualized using the GUI.

<b>Remark on the number of elements provided as output after mesh generation</b>

In providing a number of elements, GMSH aggregates the 1D elements on the boundary and the 2D elements on the interior of the domain. The number of 2D elements on the interior of the domain is thus given by the output of <i>gmsh.model.mesh.generate(2)</i> minus the output of <i>gmsh.model.mesh.generate(1)</i>. 

In [12]:
?gmsh.model.geo.addLine

```
gmsh.model.geo.addLine(startTag, endTag, tag = -1)
```

Add a straight line segment in the built-in CAD representation, between the two points with tags `startTag` and `endTag`. If `tag` is positive, set the tag explicitly; otherwise a new tag is selected automatically. Return the tag of the line.

Return an integer.

Types:

  * `startTag`: integer
  * `endTag`: integer
  * `tag`: integer


In [14]:
#..1/8: initialize gmsh 
should_finalize = Gmsh.initialize()

#..2/8: set GMSH global options 
gmsh.option.setNumber("General.Terminal",3) # make more verbose 
gmsh.option.setNumber("Mesh.Algorithm",6)   # choose mesh algorithm 
gmsh.model.add("square")                    # give name to model 

#..3/8: generate geometry 
#..set mesh density parameter 
lc = .1
#..define four points via (x,y,z) coordinates 
p1 = gmsh.model.geo.addPoint(0, 0, 0, lc, 1)
p2 = gmsh.model.geo.addPoint(1., 0,  0, lc, 2)
p3 = gmsh.model.geo.addPoint(1., 1., 0, lc, 3)
p4 = gmsh.model.geo.addPoint(0, 1., 0, lc, 4)
#..define four edges by connecting point labels pairwise  
l1 = gmsh.model.geo.addLine(1, 2, 1)
l2 = gmsh.model.geo.addLine(2, 3, 2)
l3 = gmsh.model.geo.addLine(3, 4, 3)
l4 = gmsh.model.geo.addLine(4, 1, 4)
#..define curved loop by connecting four edge labels  
loop = gmsh.model.geo.addCurveLoop([1, 2, 3, 4], 1)
#..define surface by curved loop 
surf = gmsh.model.geo.addPlaneSurface([1], 1)

#..4/8: synchronize the CAD model 
gmsh.model.geo.synchronize()

#..5/8: assign physical groups for the four boundaries and the interior 
gmsh.model.addPhysicalGroup(1, [l1], -1, "bottom")
gmsh.model.addPhysicalGroup(1, [l2], -1, "right")
gmsh.model.addPhysicalGroup(1, [l3], -1, "top")
gmsh.model.addPhysicalGroup(1, [l4], -1, "left")
gmsh.model.addPhysicalGroup(2, [surf], -1, "omega")

#..6/8: generate two-dimensional mesh 
gmsh.model.mesh.generate(1)

#..7/8: write mesh to mesh and visualize the mesh  
#..if true, write mesh to file for further processing
gmsh.option.setNumber("Mesh.Format", 16)
if (true) gmsh.write("square.msh") end 
#..if true, visualize mesh through the GUI 
if (false) gmsh.fltk.run() end 

#..8/8: finalize gmsh 
should_finalize && Gmsh.finalize()

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.00041275s, CPU 0.000329s)
Info    : 40 nodes 44 elements
Info    : Writing 'square.msh'...
Info    : Done writing 'square.msh'


false

<b>Exercises </b>: 
1. change the coordinate of one of the four points of the square and regenerate the mesh (thus practising with points, edges, closed loops and surfaces);
2. change the mesh density by changing the value of the parameter lc and regenerate the mesh. Apply different mesh density on one or more point or edges of the square (thus practising with mesh density settings); 
3. extend the code to the generation of a mesh on a pentagon (thus practising with points, edges, closed loops and surfaces); 
4. extend the code to the generation of a mesh on an L-shaped domain (thus practising with points, edges, closed loops and surfaces); 
5. extend the code to the generation of a mesh on a square with an inner square removed (thus practising the orientation of the loops);
6. extend the code to the generation of a mesh on a domain consisting of an inner and outer square (thus practising the assignment of subdomain labels);
7. extend the code to the generation of a mesh consisting of quadrilaterals (using either <i>gmsh.option.setNumber("Mesh.Algorithm", 6)</i> or <i>gmsh.model.mesh.setRecombine(2, surf)</i> ). Verify how the information written to file for triangular and quadrilateral meshes differs;
9. extend the code to the generation of a mesh consisting of trianglels or quadrilaterals with second order Lagrangian basis functions (using <i>gmsh.model.mesh.setOrder(elementOrder)</i>).  Verify how the information written to file for first and second order meshes differs; 
10. extract information on the Jacobian of the element (as a 3-by-3 matrix for a linear element), the area (one-half the determinant of the Jacobian) from GMSH using the function <i>mesh.getJacobians</i> as demonstrated in the tutorial [x6.jl](https://gitlab.onelab.info/gmsh/gmsh/blob/gmsh_4_15_0/tutorials/julia/x6.jl). Extract information on the quadrature points using the function <i>gmsh.model.mesh.getIntegrationPoints</i>; 
11. adapt the loop over the elements of the surface (see below) by a loop over the elements on (part of the) the boundary only (how to adapt the syntax (dim,tags) to multiple tags?); 
12. generate the geometry of the inductor (see below) using OpenCasCade;

## Section 3: High-Level Geometry Definition using Open-Cascasde

## Section 4: Mesh Generation (using Boundary Labels and Subdomain Labels) 

## Section 5: File Input and Output 

## Section 6: Iterative over Mesh Elements 

## Section 7: Concluding Remarks 